<a href="https://colab.research.google.com/github/Abdulkadir03/Crop-Recommender-for-Sahel-Region/blob/main/notebooks/05c_ground_truth_surveys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

05c: Ground-truth crop observations from household surveys (Nigeria, Niger, Mali, Burkina Faso and Senegal)

**Purpose:** Add independent farm-plot observations of planted crops from two World Bank Microdata Library surveys, to validate model rankings where CropHarvest has little or no coverage (Nigeria, Niger). Validation only; not used for training.

**Inputs:**
- Nigeria: National Bureau of Statistics (NBS). (2019). General Household Survey, Panel 2018-2019, Wave 4 [Dataset]. World Bank, Development Data Group. https://doi.org/10.48529/1HGW-DQ47 (ref. NGA_2018_GHSP-W4_v03_M). Downloaded from microdata.worldbank.org on <date>.
- Niger: Institut National de la Statistique (INS). (2022). Enquête Harmonisée sur le Conditions de Vie des Ménages 2018-2019 [Dataset]. World Bank, Development Data Group. https://doi.org/10.48529/GGAM-AX39 (ref. NER_2018_EHCVM_v02_M; WAEMU Commission). Downloaded from microdata.worldbank.org on <date>.
- `grid.parquet` (notebook 01) and `grid_pixel_map.parquet` (notebook 05)

**Method:**
1. Niger (EHCVM): crops per parcel from the agriculture section (crops file), cluster coordinates from the village GPS file. Nigeria (GHS-Panel W4): planted crops per plot from the post-planting visit, household coordinates from the geovariables file.
2. Survey crop names are mapped to SPAM crop codes (e.g. mil to pmil, niébé to cowp, guinea corn to sorg); unmapped crops (vegetables, trees, Bambara groundnut, etc.) are excluded from crop lists but kept in site counts.
3. Per site (Niger: village cluster; Nigeria: household) and crop: share of households growing the crop, and share of plot area it occupies (a missing plot share is treated as an equal split among crops on the plot).
4. Each site is matched to the nearest grid cell and its SPAM pixel; sites within 5 km of a grid cell are used for validation.

**Outputs** (Google Drive, not in this repo):
- `ground_truth_surveys_sites.parquet`: one row per site, with location, nearest grid cell, distance and SPAM pixel
- `ground_truth_surveys_crops.parquet`: one row per site and crop

**Notes:**
- Raw survey files are not included in this repository; request them from the World Bank Microdata Library under its terms of use.
- Household locations in these surveys may be offset or only available at area level (to be confirmed in the survey documentation), so observed crops are compared with model rankings over nearby cells or districts, not a single cell.
- Sites within 5 km of a grid cell and with a SPAM label: Niger 340 village clusters (3,169 households, 297 SPAM pixels); Nigeria 1,183 households (135 SPAM pixels). Statistics must be computed per SPAM pixel, not per household.
- Niger crops refer to the 2017/18 campaign and Nigeria's to 2018 (planting visit); SPAM represents 2019-2021.
- Niger cluster coordinates are anonymized village points and Nigeria household coordinates are modified (offset) by the survey producers, so sites are compared with model output within a few km, not exactly.
- "Beans/Cowpea" (Nigeria) is mapped to cowpea; "Mil" (Niger) and "Millet/Maiwa" (Nigeria) to pearl millet. Survey crops outside the 16-crop list (vegetables, trees, Bambara groundnut, wheat, fonio, etc.) are not evaluated.
- Crops grown at almost every Niger site (millet 97%, cowpea 95%, sorghum 87%) cannot be validated by presence; share-based comparisons are used for them.
-- Cotton, tomato, sweet potato and sugarcane have too few sites to evaluate. Yams has none: the 965 Nigerian households growing yams are all at least 43 km from the dryland grid, so yams cannot be validated. No survey coverage for Chad, Cameroon, Mauritania, Gambia, Burkina Faso, Mali or Senegal (the last three rely on CropHarvest).

In [ ]:
# @title Cell 1, setup for surveys
import os, glob, re, zipfile
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive'
SURVEYS = {'nga': ('lsms_nga_w4',     '/content/lsms_nga_w4'),
           'ner': ('ehcvm_ner_2018',  '/content/ehcvm_ner_2018'),
           'mli': ('ehcvm_mli_2018',  '/content/ehcvm_mli_2018'),
           'bfa': ('ehcvm_bfa_2018',  '/content/ehcvm_bfa_2018'),
           'sen': ('ehcvm_sen_2018',  '/content/ehcvm_sen_2018')}

def unzip_survey(key):
    folder, local = SURVEYS[key]
    os.makedirs(local, exist_ok=True)
    for z in glob.glob(f'{BASE}/crop_project/raw/{folder}/*.zip'):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(local)
    return local

grid = pd.read_parquet(f'{BASE}/crop_project/grid.parquet')
gm = pd.read_parquet(f'{BASE}/crop_project/grid_pixel_map.parquet')
for k in SURVEYS:
    print(k, '->', len(glob.glob(f'{unzip_survey(k)}/**/*.dta', recursive=True)), '.dta files')

In [ ]:
# @title Cell 2: catalogue of the Nigeria files
def catalogue(key):
    local = SURVEYS[key][1]
    rows = []
    for f in sorted(glob.glob(f'{local}/**/*.dta', recursive=True)):
        try:
            d = pd.read_stata(f, convert_categoricals=False)
        except Exception as e:
            rows.append({'file': os.path.basename(f), 'error': str(e)[:60]})
            continue
        cols = list(d.columns)
        rows.append({'file': os.path.basename(f), 'rows': len(d), 'cols': len(cols),
                     'hhid': 'hhid' in cols, 'plotid': 'plotid' in cols, 'cropcode': 'cropcode' in cols,
                     'geo': any(c.lower().startswith(('lat', 'lon')) for c in cols),
                     'first_vars': ', '.join(cols[:6])})
    return pd.DataFrame(rows)

pd.set_option('display.width', 250); pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 70)
print(catalogue('nga').to_string(index=False))

In [ ]:
# @title Cell 3, inspect the Niger files
NEED = ['s16c_me_ner2018', 's16a_me_ner2018', 'ehcvm_menage_ner2018',
        'grappe_gps_ner2018', 's03_co_ner2018', 'ehcvm_ponderations_ner2018']
KEY = re.compile(r'(?i)superf|surface|area|taille|parcel|plot|irrig|sol|cultur|crop|produit|recolt|id|grappe|menage|region|depart|commune|milieu')
LOC = SURVEYS['ner'][1]

def find(name):
    hits = glob.glob(f'{LOC}/**/{name}.dta', recursive=True)
    return hits[0] if hits else None

for n in NEED:
    f = find(n)
    if not f:
        print(f'\n== {n}: NOT FOUND'); continue
    with pd.io.stata.StataReader(f) as r:
        labs = r.variable_labels()
    d = pd.read_stata(f, convert_categoricals=False)
    print(f'\n== {n}: {len(d)} rows x {d.shape[1]} cols')
    keep = [c for c in d.columns if KEY.search(c) or KEY.search(str(labs.get(c, '')))]
    show = list(dict.fromkeys(d.columns[:8].tolist() + keep))[:45] if len(d.columns) > 60 else d.columns.tolist()
    for c in show:
        print(f'   {c:16} {str(labs.get(c, ""))[:75]}')
    if n == 'grappe_gps_ner2018':
        print(d.head(5).to_string())

f = find('s16c_me_ner2018')
if f:
    dc = pd.read_stata(f)
    with pd.io.stata.StataReader(f) as r:
        labs = r.variable_labels()
    cc = [c for c in dc.columns if re.search(r'(?i)cultur|crop', str(labs.get(c, '')))]
    print('\nCrop-name columns:', cc)
    if cc:
        print(dc[cc[0]].value_counts().head(40))

In [ ]:
# @title Cell 4: Niger crops and clusters
from sklearn.neighbors import BallTree

def load(name, loc, cats=False):
    hits = glob.glob(f'{loc}/**/{name}.dta', recursive=True)
    return pd.read_stata(hits[0], convert_categoricals=cats) if hits else None

def labels_of(name, loc):
    hits = glob.glob(f'{loc}/**/{name}.dta', recursive=True)
    with pd.io.stata.StataReader(hits[0]) as r:
        return r.variable_labels()

_tree = BallTree(np.radians(grid[['lat', 'lon']].to_numpy('float64')), metric='haversine')
def dist_to_grid(df, latcol, loncol):
    d, _ = _tree.query(np.radians(df[[latcol, loncol]].to_numpy('float64')), k=1)
    return d[:, 0] * 6371.0

LOCN = SURVEYS['ner'][1]
c = load('s16c_me_ner2018', LOCN, cats=True)
print(c[['grappe', 'menage', 's16cq02', 's16cq03', 's16cq04', 's16cq07', 's16cq08', 's16cq10']].head(8).to_string())
print('\nCrop codes (s16cq04), top 60:'); print(c['s16cq04'].value_counts(dropna=False).head(60))
print('\nCultivation system (s16cq07):'); print(c['s16cq07'].value_counts(dropna=False))
print('\nShare of parcel occupied (s16cq08):'); print(c['s16cq08'].value_counts(dropna=False).head(15))
print('\nClusters with crop data:', c.grappe.nunique(),
      '| households:', c[['grappe', 'menage']].drop_duplicates().shape[0])

g = load('grappe_gps_ner2018', LOCN).rename(
    columns={'coordonnes_gps__Latitude': 'lat', 'coordonnes_gps__Longitude': 'lon'})
g = g.dropna(subset=['lat', 'lon']).copy()
print('\nClusters with GPS:', len(g),
      '| crop clusters that have GPS:', round(float(c.grappe.drop_duplicates().isin(g.grappe).mean()), 3))
cl = g[g.grappe.isin(c.grappe.unique())].copy()
cl['km'] = dist_to_grid(cl, 'lat', 'lon')
print('Crop clusters within 2 / 5 / 10 km of a grid cell:', [int((cl.km <= r).sum()) for r in (2, 5, 10)], 'of', len(cl))

In [ ]:
# @title Cell 5: Nigeria crops and household coordinates
LOCG = SURVEYS['nga'][1]

def show_labels(name, loc):
    labs = labels_of(name, loc)
    d = load(name, loc)
    print(f'\n== {name}: {len(d)} rows x {d.shape[1]} cols')
    for col in d.columns:
        print(f'   {col:18} {str(labs.get(col, ""))[:80]}')

hg = load('nga_householdgeovars_y4', LOCG)
ll = [col for col in hg.columns if col.lower().startswith(('lat', 'lon'))]
print('Household geovars:', hg.shape, '| coordinate columns:', ll)
print(hg[ll].describe().round(3))

show_labels('sect11f_plantingw4', LOCG)
show_labels('sect11a1_plantingw4', LOCG)
show_labels('secta3i_harvestw4', LOCG)

cp = load('sect11f_plantingw4', LOCG, cats=True)
print('\nCrop codes (cropcode), top 45:'); print(cp['cropcode'].value_counts(dropna=False).head(45))
print('Households with planted-crop rows:', cp.hhid.nunique(),
      '| plots:', cp[['hhid', 'plotid']].drop_duplicates().shape[0])

latc = [col for col in ll if col.lower().startswith('lat')][0]
lonc = [col for col in ll if col.lower().startswith('lon')][0]
h = hg[['hhid', latc, lonc]].dropna()
h = h[h.hhid.isin(cp.hhid.unique())].copy()
h['km'] = dist_to_grid(h, latc, lonc)
print('\nHouseholds with crops and coordinates:', len(h),
      '| within 2 / 5 / 10 km of a grid cell:', [int((h.km <= r).sum()) for r in (2, 5, 10)])

In [ ]:
# @title Cell 6: build the survey tables
#@title Build survey ground-truth tables (Niger, Nigeria) { display-mode: "both" }
# One site table and one site x crop table per survey, then matched to the nearest grid cell.
def nearest_grid(df, latcol, loncol):
    d, i = _tree.query(np.radians(df[[latcol, loncol]].to_numpy('float64')), k=1)
    out = df.copy()
    out['nearest_km'] = d[:, 0] * 6371.0
    out['nearest_grid_id'] = grid.grid_id.to_numpy()[i[:, 0]]
    return out

def plot_weights(df, plot_keys, pct_col):
    """Share of the plot occupied by the crop; missing share = equal split among crops on the plot."""
    pct = pd.to_numeric(df[pct_col], errors='coerce')
    n = df.groupby(plot_keys)[plot_keys[0]].transform('size')
    return pct.fillna(100.0 / n).clip(0, 100)

def build(df, site_keys, hh_keys, wt_col):
    key = list(dict.fromkeys(site_keys + hh_keys))
    sites = df[key].drop_duplicates().groupby(site_keys).size().rename('n_hh').reset_index()
    tot = df.groupby(site_keys)[wt_col].sum().rename('tot_w').reset_index()
    m = df.dropna(subset=['crop_code'])
    nh = (m[key + ['crop_code']].drop_duplicates().groupby(site_keys + ['crop_code'])
          .size().rename('n_hh_crop').reset_index())
    wc = m.groupby(site_keys + ['crop_code'])[wt_col].sum().rename('w_crop').reset_index()
    crops = nh.merge(wc, on=site_keys + ['crop_code']).merge(sites, on=site_keys).merge(tot, on=site_keys)
    crops['share_hh'] = crops.n_hh_crop / crops.n_hh
    crops['weight_share'] = crops.w_crop / crops.tot_w
    return sites, crops[site_keys + ['crop_code', 'n_hh_crop', 'share_hh', 'weight_share']]

# ---------------- Niger (EHCVM 2018-19, crops of the 2017/18 campaign) ----------------
LOCN = SURVEYS['ner'][1]
NER_MAP = {'Mil': 'pmil', 'Niébé': 'cowp', 'Sorgho': 'sorg', 'Arachide': 'grou', 'Sésame': 'sesa',
           'Riz Paddy': 'rice', 'Oignon': 'onio', 'Maïs': 'maiz', 'Tomate': 'toma',
           'Manioc': 'cass', 'Patate douce': 'swpo', 'Coton': 'cott'}
cn = load('s16c_me_ner2018', LOCN, cats=True)
cn['crop_code'] = cn['s16cq04'].astype(str).str.strip().map(NER_MAP)
cn['w'] = plot_weights(cn, ['grappe', 'menage', 's16cq02', 's16cq03'], 's16cq08')
sN, cN = build(cn, ['grappe'], ['menage'], 'w')
gps = (load('grappe_gps_ner2018', LOCN)
       .rename(columns={'coordonnes_gps__Latitude': 'lat', 'coordonnes_gps__Longitude': 'lon'})
       [['grappe', 'lat', 'lon']].dropna())
sN = sN.merge(gps, on='grappe', how='inner')
for t in (sN, cN):
    t['site_id'] = 'NER_' + t.grappe.astype(int).astype(str)
sN = nearest_grid(sN.drop(columns='grappe'), 'lat', 'lon'); cN = cN.drop(columns='grappe')
sN['country'], sN['season'] = 'Niger', '2017/18'

# ---------------- Nigeria (GHS-Panel W4, planting visit 2018) ----------------
LOCG = SURVEYS['nga'][1]
NGA_CODE = {1010: 'cowp', 1020: 'cass', 1060: 'grou', 1070: 'sorg', 1080: 'maiz', 1100: 'pmil',
            1110: 'rice', 1120: 'yams', 1121: 'yams', 1122: 'yams', 1123: 'yams', 1124: 'yams',
            2040: 'sesa', 2130: 'onio', 2181: 'swpo', 2220: 'soyb', 2230: 'sugc', 2260: 'toma'}
cg = load('sect11f_plantingw4', LOCG, cats=True)
lab = cg['cropcode'].astype(str)
cg['crop_code'] = lab.str.extract(r'^(\d+)')[0].astype(float).map(NGA_CODE)
cg.loc[lab.str.upper().str.contains('COTTON'), 'crop_code'] = 'cott'
cg['w'] = plot_weights(cg, ['hhid', 'plotid'], 's11fq1')
sG, cG = build(cg, ['hhid'], ['hhid'], 'w')
hgeo = (load('nga_householdgeovars_y4', LOCG)
        .rename(columns={'lat_dd_mod': 'lat', 'lon_dd_mod': 'lon'})[['hhid', 'lat', 'lon']].dropna())
sG = sG.merge(hgeo, on='hhid', how='inner')
for t in (sG, cG):
    t['site_id'] = 'NGA_' + t.hhid.astype(int).astype(str)
sG = nearest_grid(sG.drop(columns='hhid'), 'lat', 'lon'); cG = cG.drop(columns='hhid')
sG['country'], sG['season'] = 'Nigeria', '2018'

# ---------------- Combine, add SPAM pixel, save ----------------
sites = pd.concat([sN, sG], ignore_index=True)
sites = sites.merge(gm[['grid_id', 'px', 'py', 'has_label']].rename(columns={'grid_id': 'nearest_grid_id'}),
                    on='nearest_grid_id', how='left')
sites['within_5km'] = sites.nearest_km <= 5
crops = pd.concat([cN, cG], ignore_index=True)
crops = crops[crops.site_id.isin(sites.site_id)]

sites.to_parquet(f'{BASE}/crop_project/ground_truth_surveys_sites.parquet', index=False)
crops.to_parquet(f'{BASE}/crop_project/ground_truth_surveys_crops.parquet', index=False)

use = sites[sites.within_5km & (sites.has_label == True)].copy()
use['pixel'] = use.px.astype(str) + '_' + use.py.astype(str)
print(use.groupby('country').agg(sites=('site_id', 'nunique'), households=('n_hh', 'sum'),
                                 spam_pixels=('pixel', 'nunique')))
cu = crops.merge(use[['site_id', 'country', 'pixel']], on='site_id')
print('\nSites with the crop (within 5 km, with a SPAM label):')
print(cu.groupby(['crop_code', 'country']).site_id.nunique().unstack().fillna(0).astype(int))
print('\nDistinct SPAM pixels with the crop:')
print(cu.groupby(['crop_code', 'country']).pixel.nunique().unstack().fillna(0).astype(int))
drive.flush_and_unmount(); drive.mount('/content/drive')